# Coder, phase 2: train Qwen2.5-Coder-14B to fix bugs (QLoRA on Kaggle's free T4)

**What it does:**
1. Downloads real bug-fix commits from GitHub: **CommitPackFT (Python)** (`bigcode/commitpackft`). It stops if the
   dataset's licence is not an open one, and keeps only commits from permissively licensed repositories.
2. Keeps small bug fixes (the message says fix/bug/error/...; at most 20 lines changed) and **drops every commit
   that shares text with the test tasks** (any 10 words in a row, or the same function signature), so the score
   is honest.
3. Scores the model as it is on **HumanEvalFix (Python)** (164 buggy functions with hidden tests): the BEFORE score.
4. Trains a small LoRA add-on (QLoRA, 4-bit) on the commits: the model sees the file before the fix and learns to
   write the file after it.
5. Scores it again: the AFTER score. Phase 1's best was qwen3-coder:30b with 73%.

Everything goes to `coder_train.json` (Output tab) after each step, so nothing is lost if it stops.
Time: about 5–8 hours (each score 20–60 minutes, at most 2 hours; training at most 4 hours). `SMOKE = True`: about 20–30 minutes.

**Settings (right side):** Accelerator **GPU T4 x2**, Internet **on**. Then **Run all**.

**Or with no clicks:** on GitHub, **Actions → Coder model on Kaggle → Run workflow → action train**, or ask Claude.

In [ ]:
#@title 1. Check the GPU
import os, shutil, subprocess, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle gives two GPUs; training uses one
# nvidia-smi is not always installed, so also ask torch (in its own process: nothing is imported here yet).
cuda = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.is_available())"],
                      capture_output=True, text=True).stdout.strip() == "True"
if not (cuda or shutil.which("nvidia-smi")):
    # What the machine has, to see why there is no GPU (no secret values printed).
    print("nvidia devices:", [d for d in os.listdir("/dev") if d.startswith("nvidia")])
    print("Kaggle settings:", {k: v for k, v in os.environ.items()
                               if k.startswith("KAGGLE") and any(w in k for w in ("ACCELERATOR", "GPU", "MACHINE", "RUN_TYPE"))})
    raise RuntimeError("No GPU. Colab: Runtime > Change runtime type > T4 GPU > Save, then Run all. "
                       "Kaggle: verify your phone, then turn on GPU T4 once in the notebook editor (Settings > Accelerator).")
print(subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.get_device_name(0))"],
                     capture_output=True, text=True).stdout or "GPU found")

In [ ]:
#@title 2. Install the training tools (about 3–5 minutes)
!pip install -q unsloth

In [ ]:
#@title 3. Settings: base model, how many examples, time limits
import json, os, random, re, time

# The model to train: 4-bit Qwen2.5-Coder-14B-Instruct (Apache 2.0), ready for Unsloth.
BASE = "unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit"
# How many bug-fix commits to train on at most. Claude and GitHub Actions fill this line.
MAX_TRAIN = 2000
MAX_LEN = 2048  # tokens; a longer example (question + answer) is left out
# Passes over the training examples. Claude and GitHub Actions fill this line.
EPOCHS = 1
EVAL_LIMIT = None  # how many HumanEvalFix tasks to score (None = all 164)
# True = a quick check of the whole run (40 examples, 8 steps, 8 tasks; about 20-30 minutes). Claude and GitHub
# Actions fill this line.
SMOKE = False
TRAIN_MINUTES = 240  # training stops after this, so the whole run ends inside Kaggle's 12 hours
EVAL_MINUTES = 120  # each score (before, after) stops after this
EVAL_BATCH = 4  # tasks answered together (halved by itself when the GPU runs out of memory)
MAX_NEW_TOKENS = 1024  # longest reply
MAX_STEPS = None  # None: as many steps as the examples and EPOCHS need
LOGGING_STEPS = 10
SEED = 3407
WORKING_DIR = "/kaggle/working"  # coder_train.json and the LoRA are saved here (the notebook's Output tab)
LORA_DIR = os.path.join(WORKING_DIR, "coder-lora")
if SMOKE:
    MAX_TRAIN, MAX_STEPS, EVAL_LIMIT, LOGGING_STEPS = 40, 8, 8, 1

In [ ]:
#@title 4. Helpers (no need to read; built from coder/fixeval.py and coder/fixtrain.py)
"""HumanEvalFix (Python) through Ollama: show a model a buggy function, run its fix against the hidden tests.

Plain Python (no packages to install). coder/build.py puts this whole file into coder_eval.ipynb, and
coder/run_eval.py uses table() to print the results, so the notebook and the report agree.
"""

import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.request

BENCHMARK = "HumanEvalFix (Python)"
DATASET = "bigcode/humanevalpack"  # config "python", split "test": 164 tasks
LICENCE = "MIT"  # the HumanEvalPack licence (bigcode/humanevalpack on Hugging Face)
DATASET_URL = "https://huggingface.co/datasets/bigcode/humanevalpack/resolve/main/data/python/data/humanevalpack.jsonl"
NEEDED = ("task_id", "declaration", "buggy_solution", "test", "entry_point")
INSTRUCTION = "Fix bugs in {entry_point}. Reply with the full corrected function in one ```python block."
SETTING = "HumanEvalFix, no tests shown; greedy (temperature 0); pass@1 = tests pass with the one answer"
TEST_TIMEOUT = 10  # seconds for one task's tests
# What the standard HumanEval runners put above the code (standard library only).
IMPORTS = """import math
import re
import sys
import copy
import datetime
import itertools
import collections
import heapq
import functools
import hashlib
import string
from typing import *
from collections import *
"""


# ---- the tasks ----

def load_tasks():
    """All HumanEvalFix Python tasks (a list of dicts) and where they came from."""
    try:
        with urllib.request.urlopen(DATASET_URL, timeout=120) as resp:
            text = resp.read().decode("utf-8")
        rows, source = [json.loads(line) for line in text.splitlines() if line.strip()], DATASET_URL
    except Exception as first:  # noqa: BLE001 - try the datasets library next
        rows, source, problems = None, None, [f"{DATASET_URL}: {first!r}"]
        for revision in ("refs/convert/parquet", None):
            try:
                from datasets import load_dataset

                rows = [dict(r) for r in load_dataset(DATASET, "python", split="test", revision=revision)]
                source = f"datasets.load_dataset({DATASET!r}, 'python', split='test', revision={revision!r})"
                break
            except Exception as e:  # noqa: BLE001
                problems.append(f"datasets (revision {revision}): {e!r}")
        if rows is None:
            raise RuntimeError("Could not download HumanEvalFix: " + " | ".join(problems)) from None
    if not rows:
        raise RuntimeError(f"HumanEvalFix from {source} has no tasks")
    missing = [f for f in NEEDED if f not in rows[0]]
    if missing:
        raise RuntimeError(f"HumanEvalFix rows have no {missing}; they have {sorted(rows[0])}")
    return rows, source


def prompt_for(task):
    """The message to the model: the buggy function, then the instruction (no tests shown)."""
    buggy = (task["declaration"] + task["buggy_solution"]).strip("\n")
    return f"```python\n{buggy}\n```\n\n" + INSTRUCTION.format(entry_point=task["entry_point"])


# ---- the reply ----

THINKING = re.compile(r"<think(?:ing)?>.*?</think(?:ing)?>", re.DOTALL)
BLOCK = re.compile(r"```[ \t]*([\w+.-]*)[^\n]*\n(.*?)```", re.DOTALL)
OPEN_BLOCK = re.compile(r"```[ \t]*(?:python3?|py)?[ \t]*\n(.*)\Z", re.DOTALL)  # cut off before the closing ```


def without_thinking(reply):
    """The reply without <think>...</think> parts (a thinking model's notes before its answer)."""
    reply = THINKING.sub("", reply or "")
    if "</think>" in reply:  # the opening tag was in the prompt template, only the closing one came back
        reply = reply.rsplit("</think>", 1)[1]
    return reply


def extract_code(reply):
    """The code to test: the last ```python block (or the last block of any kind; or an unclosed one at the end;
    or else the whole reply)."""
    reply = without_thinking(reply)
    blocks = BLOCK.findall(reply)
    python = [body for lang, body in blocks if lang.lower() in ("python", "python3", "py", "")]
    if python or blocks:
        return (python or [body for _, body in blocks])[-1].strip("\n") + "\n"
    cut_off = OPEN_BLOCK.search(reply)
    return (cut_off.group(1) if cut_off else reply).strip("\n") + "\n"


def program_for(code, task):
    """The file that is run: imports, the model's code, the dataset's tests and the call that runs them."""
    test = task["test"]
    call = f"check({task['entry_point']})"
    if not re.search(rf"^check\(\s*{re.escape(task['entry_point'])}\s*\)", test, re.MULTILINE):
        test = test.rstrip("\n") + f"\n\n{call}\n"
    return "\n".join((IMPORTS, task.get("import") or "", code, "", test))


def run_program(program, timeout=None):
    """Run it in a new Python process. (passed, why): passed when it ends with exit code 0 within the time
    (TEST_TIMEOUT seconds unless timeout is given)."""
    timeout = timeout or TEST_TIMEOUT
    with tempfile.TemporaryDirectory() as folder:
        path = os.path.join(folder, "task.py")
        with open(path, "w", encoding="utf-8") as fh:
            fh.write(program)
        try:
            done = subprocess.run([sys.executable, path], cwd=folder, stdin=subprocess.DEVNULL, capture_output=True,
                                  text=True, timeout=timeout, check=False)
        except subprocess.TimeoutExpired:
            return False, f"timeout after {timeout} s"
    if done.returncode == 0:
        return True, "passed"
    lines = [line for line in (done.stderr or done.stdout).splitlines() if line.strip()]
    return False, (lines[-1].strip()[:200] if lines else f"exit code {done.returncode}")


# ---- Ollama ----

def post(url, path, body, timeout=600):
    request = urllib.request.Request(url + path, json.dumps(body).encode(), {"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=timeout) as resp:
            return json.load(resp)
    except urllib.error.HTTPError as e:  # Ollama says why in the body: {"error": "..."}
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode('utf-8', 'replace')[:500]}") from None


def pull(url, model, log=print):
    """Download a model (like `ollama pull`). Its size in GB. Prints each new step."""
    request = urllib.request.Request(url + "/api/pull", json.dumps({"model": model, "stream": True}).encode(),
                                     {"Content-Type": "application/json"})
    sizes, last = {}, None
    try:
        with urllib.request.urlopen(request, timeout=900) as resp:
            for line in resp:
                if not line.strip():
                    continue
                step = json.loads(line)
                if step.get("error"):
                    raise RuntimeError(step["error"])
                if step.get("digest") and step.get("total"):
                    sizes[step["digest"]] = step["total"]
                status = re.sub(r"pulling [0-9a-f]{12,}", "pulling a part", step.get("status", ""))
                if status != last:
                    log(f"   {model}: {status}")
                    last = status
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode('utf-8', 'replace')[:500]}") from None
    if last != "success":
        raise RuntimeError(f"the download did not finish (last step: {last})")
    return round(sum(sizes.values()) / 1e9, 1)


def in_memory(url, model):
    """The model as Ollama holds it now (from /api/ps: size, size_vram, ...), or None when it is not loaded."""
    with urllib.request.urlopen(url + "/api/ps", timeout=10) as resp:
        models = json.load(resp).get("models") or []
    names = {model, model + ":latest"}
    return next((m for m in models if m.get("name") in names or m.get("model") in names), None)


def processor(url, model):
    """Where the model runs, as `ollama ps` says it: "100% GPU", "100% CPU", "25%/75% CPU/GPU", or None."""
    try:
        m = in_memory(url, model)
    except Exception:  # noqa: BLE001
        return None
    if not m or not m.get("size"):
        return None
    cpu = round(100 * (m["size"] - (m.get("size_vram") or 0)) / m["size"])
    return "100% GPU" if cpu <= 0 else "100% CPU" if cpu >= 100 else f"{cpu}%/{100 - cpu}% CPU/GPU"


def unload(url, model):
    """Take the model out of memory (keep_alive 0) and wait until it is gone."""
    post(url, "/api/generate", {"model": model, "keep_alive": 0}, timeout=120)
    for _ in range(60):
        if in_memory(url, model) is None:
            return
        time.sleep(1)


def delete(url, model):
    """Delete the downloaded model (like `ollama rm`), to free the disk for the next one."""
    request = urllib.request.Request(url + "/api/delete", json.dumps({"model": model}).encode(),
                                     {"Content-Type": "application/json"}, method="DELETE")
    with urllib.request.urlopen(request, timeout=120):
        pass


def text_of(command):
    """What a command printed ("" when it is missing or fails), e.g. `ollama ps` or nvidia-smi."""
    try:
        return subprocess.run(command, shell=True, capture_output=True, text=True, timeout=60,
                              check=False).stdout.strip()
    except Exception:  # noqa: BLE001
        return ""


def think_for(model, think):
    """The "think" setting for this model ({"gpt-oss": "low"} → "low" for gpt-oss:20b), or None."""
    return next((level for prefix, level in (think or {}).items() if model.startswith(prefix)), None)


def ask(url, model, prompt, options, think=None, timeout=600):
    """One chat request (no system message). The final answer only: a thinking model's thinking is kept apart."""
    body = {"model": model, "messages": [{"role": "user", "content": prompt}], "stream": False, "options": options,
            "keep_alive": "30m", **({"think": think} if think else {})}
    started = time.time()
    data = post(url, "/api/chat", body, timeout=timeout)
    message = data.get("message") or {}
    return {"reply": message.get("content") or "", "thinking_chars": len(message.get("thinking") or ""),
            "seconds": round(time.time() - started, 2), "tokens": data.get("eval_count") or 0,
            "token_seconds": (data.get("eval_duration") or 0) / 1e9, "done_reason": data.get("done_reason")}


# ---- one model ----

def disk_free_gb(path):
    try:
        return round(shutil.disk_usage(path).free / 1e9, 1)
    except OSError:
        return None


def evaluate_model(url, model, tasks, entry, minutes, options, think=None, thinking_num_predict=None,
                   save=lambda: None, log=print, models_dir="/tmp"):
    """Download, load and score one model, filling entry (a dict in the results) as it goes, so a failure part-way
    keeps everything done until then. Stops when `minutes` have passed (counting the download and loading)."""
    deadline = time.time() + minutes * 60
    level = think_for(model, think)
    options = {**options, **({"num_predict": thinking_num_predict} if level and thinking_num_predict else {})}
    entry.update(model=model, think=level, options=options, total=len(tasks), done=0, passed=0, pass_at_1=None,
                 rate_done=None, pulled=False, pull_seconds=None, size_gb=None, load_seconds=None, processor=None,
                 ollama_ps=None, gpu_memory=None, avg_seconds=None, tokens_per_s=None, cut_off=0, no_code_block=0,
                 stopped_by_time=False, minutes=None, disk_free_gb_before=disk_free_gb(models_dir), errors=[],
                 tasks=[])
    started = time.time()

    log(f"\n== {model}: download")
    entry["size_gb"] = pull(url, model, log)
    entry.update(pulled=True, pull_seconds=round(time.time() - started, 1))
    log(f"   {model}: {entry['size_gb']} GB in {entry['pull_seconds']} s")
    save()

    loading = time.time()
    post(url, "/api/generate", {"model": model, "options": options, "keep_alive": "30m"}, timeout=900)  # only load
    entry["load_seconds"] = round(time.time() - loading, 1)
    log(f"   {model}: loaded in {entry['load_seconds']} s")
    save()

    failures_in_a_row = 0
    for i, task in enumerate(tasks, 1):
        if time.time() > deadline:
            entry["stopped_by_time"] = True
            log(f"   {model}: stopped at {minutes} minutes, after {entry['done']} of {len(tasks)} tasks")
            break
        record = {"task_id": task["task_id"], "passed": False}
        try:
            answer = ask(url, model, prompt_for(task), options, level)
        except Exception as e:  # noqa: BLE001 - count it as a fail and go on
            record["why"] = f"request failed: {e!r}"[:300]
            entry["errors"].append(f"{task['task_id']}: {record['why']}")
            failures_in_a_row += 1
        else:
            failures_in_a_row = 0
            code = extract_code(answer["reply"])
            has_block = "```" in without_thinking(answer["reply"])
            passed, why = run_program(program_for(code, task))
            if not answer["reply"].strip():
                why = "no answer" + (" (cut off while thinking)" if answer["done_reason"] == "length" else "")
            record.update(answer, passed=passed, why=why, code_block=has_block)
            entry["cut_off"] += answer["done_reason"] == "length"
            entry["no_code_block"] += not has_block
        entry["tasks"].append(record)
        entry["done"] += 1
        entry["passed"] += record["passed"]
        log(f"   {i:3d}/{len(tasks)} {task['task_id']:16s} {'pass' if record['passed'] else 'FAIL'}  "
            f"{record.get('seconds', '-')} s  {'' if record['passed'] else record['why'][:80]}")
        if i == 1:  # where it runs, once it has answered (`ollama ps`)
            entry.update(processor=processor(url, model), ollama_ps=text_of("ollama ps") or None,
                         gpu_memory=text_of("nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader")
                         .replace("\n", "; ") or None)
            log(f"   {model} runs on: {entry['processor']}")
        summarise(entry, started)
        if i % 10 == 0:
            save()
        if failures_in_a_row >= 3:
            entry["errors"].append("3 requests in a row failed: stopped this model")
            log(f"   {model}: 3 requests in a row failed, going on to the next model")
            break
    summarise(entry, started)
    save()
    return entry


def summarise(entry, started):
    """The totals of one model from its task records."""
    answered = [t for t in entry["tasks"] if "seconds" in t]
    token_seconds = sum(t["token_seconds"] for t in answered)
    entry["pass_at_1"] = round(entry["passed"] / entry["total"], 4) if entry["total"] and \
        entry["done"] == entry["total"] else None
    entry["rate_done"] = round(entry["passed"] / entry["done"], 4) if entry["done"] else None
    entry["avg_seconds"] = round(sum(t["seconds"] for t in answered) / len(answered), 2) if answered else None
    entry["tokens_per_s"] = round(sum(t["tokens"] for t in answered) / token_seconds, 1) if token_seconds else None
    entry["minutes"] = round((time.time() - started) / 60, 1)


def tidy_up(url, model, entry, delete_after=True, log=print):
    """Unload the model (and delete it, to free the disk). Problems are recorded, never raised."""
    for step, do in (("unload", unload), ("delete", delete if delete_after else None)):
        if do is None:
            continue
        try:
            do(url, model)
        except Exception as e:  # noqa: BLE001 - the next model must still run
            entry.setdefault("errors", []).append(f"{step}: {e!r}"[:300])
            log(f"   {model}: {step} failed: {e!r}")


# ---- the report ----

def table(results):
    """The results as text lines: one row per model."""
    lines = [f"{'model':24s} {'passed':>13s} {'%':>5s} {'s/task':>7s} {'tok/s':>6s}  {'ran on':16s} note"]
    for m in results.get("models") or []:
        done, total = m.get("done") or 0, m.get("total") or 0
        rate = m.get("pass_at_1") if m.get("pass_at_1") is not None else m.get("rate_done")
        note = []
        if not m.get("pulled"):
            note.append("download failed")
        elif m.get("load_seconds") is None:
            note.append("did not load")
        if m.get("stopped_by_time"):
            note.append(f"stopped at the time limit after {done} of {total} tasks")
        elif done and done < total:
            note.append(f"stopped after {done} of {total} tasks")
        if m.get("cut_off"):
            note.append(f"{m['cut_off']} cut off")
        if m.get("think"):
            note.append(f"think {m['think']}")
        if m.get("errors"):
            note.append(f"{len(m['errors'])} error" + ("s" if len(m["errors"]) > 1 else ""))
        passed = f"{m.get('passed') or 0}/{done}" + ("" if done == total else f" of {total}")
        percent = "-" if rate is None else f"{rate:.0%}"
        lines.append(f"{m.get('model', '?'):24s} {passed:>13s} {percent:>5s} {m.get('avg_seconds') or '-':>7} "
                     f"{m.get('tokens_per_s') or '-':>6}  {m.get('processor') or '-':16s} {'; '.join(note)}".rstrip())
    lines.append("(% = passed of all tasks when all were done, else of the tasks done; greedy pass@1)")
    return lines


"""Phase 2: train a coding model on open bug-fix commits, then score it on HumanEvalFix again.

Plain Python (no packages needed to import it). coder/build_train.py puts this whole file, after coder/fixeval.py,
into coder_train.ipynb; coder/run_train.py uses report_lines() to print the results, so the notebook and the report
agree. The notebook itself adds the parts that need the GPU (Unsloth, torch, trl).
"""

import difflib
import json
import random
import re
import time

try:  # outside the notebook; in the notebook, fixeval.py's code is in the same cell, above this
    from coder.fixeval import extract_code, program_for, prompt_for, run_program, without_thinking
except ImportError:
    pass

TRAIN_DATASET = "bigcode/commitpackft"  # config "python": real commits from GitHub, with the file before and after
TRAIN_FILE = "data/python/data.jsonl"
COMMIT_FIELDS = ("old_contents", "new_contents", "subject")
# The dataset's licence (from its card) must be one of these, or the notebook stops.
OPEN_LICENCES = ("mit", "apache-2.0", "cc-by-4.0", "cc0-1.0", "cc0", "odc-by")  # and every "bsd-..."
# Each row's own licence (the repository it came from), when the row has one.
ROW_LICENCES = ("mit", "apache-2.0", "bsd-2-clause", "bsd-3-clause", "isc", "unlicense", "cc0-1.0")
MAX_CHANGED_LINES = 20
NGRAM = 10  # a training row that shares this many words in a row with a HumanEvalFix task is dropped
DECONTAM_FIELDS = ("prompt", "declaration", "buggy_solution", "canonical_solution", "test")
TRAIN_INSTRUCTION = "Fix bugs: {subject}. Reply with the full corrected code in one ```python block."
EVAL_SETTING = ("HumanEvalFix, no tests shown; greedy; the 4-bit (bitsandbytes) model through Unsloth/transformers "
                "generate. Phase 1 used Ollama's Q4_K_M file, so 'before' may differ a little from its 64%; "
                "before and after use the same setup, so their comparison is fair.")
NO_EXPORT = ("No GGUF/Ollama file in this phase: merging the LoRA into a 14B model needs more memory and disk than "
             "Kaggle gives safely. The LoRA add-on is saved (coder-lora).")
PHASE1 = {"qwen3-coder:30b": 120 / 164, "qwen2.5-coder:14b": 105 / 164}  # Ollama Q4_K_M, 2026-10-15
TARGET = ("qwen3-coder:30b", 120 / 164)  # the best model of phase 1: the score to beat

FIX = re.compile(r"\b(fix(e[sd]|ing)?|hotfix(es)?|bugs?|bugfix(es)?|errors?|wrong(ly)?|incorrect(ly)?"
                 r"|crash(es|ed|ing)?|typos?|exceptions?)\b", re.IGNORECASE)
NOT_A_FIX = re.compile(r"\b(merge[sd]?|revert(s|ed)?|bump(s|ed)?|version|readme|docs?|documentation)\b", re.IGNORECASE)
TEST_FILE = re.compile(r"(^|/)(tests?/|test_[^/]*\.py$|[^/]*_tests?\.py$|conftest\.py$)")


# ---- the training data ----

def open_licence(name) -> bool:
    name = str(name or "").strip().lower()
    return name in OPEN_LICENCES or name.startswith("bsd")


def check_licence(value) -> list[str]:
    """The licence(s) from a dataset card, as a list; RuntimeError unless there is one and all are open."""
    names = [value] if isinstance(value, str) else list(value or [])
    names = [str(n).strip().lower() for n in names if str(n).strip()]
    if not names:
        raise RuntimeError(f"{TRAIN_DATASET}: its card names no licence, so it is not used")
    closed = [n for n in names if not open_licence(n)]
    if closed:
        raise RuntimeError(f"{TRAIN_DATASET}: licence {closed} is not an open licence {OPEN_LICENCES} or bsd-*; "
                           "not used")
    return names


def dataset_licence(repo=TRAIN_DATASET) -> dict:
    """The dataset's licence from its Hugging Face card (refuses a closed or missing one)."""
    from huggingface_hub import DatasetCard, HfApi

    value, source = None, None
    try:
        card = HfApi().dataset_info(repo).card_data
        value = (card.to_dict() if hasattr(card, "to_dict") else dict(card or {})).get("license")
        source = "dataset_info card_data"
    except Exception as e:  # noqa: BLE001 - try the card file itself
        print("dataset_info:", repr(e))
    if not value:
        value, source = DatasetCard.load(repo, repo_type="dataset").data.to_dict().get("license"), "DatasetCard"
    return {"dataset": repo, "licence": check_licence(value), "from": source}


def load_commits(repo=TRAIN_DATASET, file=TRAIN_FILE):
    """The Python commits of CommitPackFT (a list of dicts) and where they came from."""
    problems = []
    try:
        from huggingface_hub import hf_hub_download

        path = hf_hub_download(repo, file, repo_type="dataset")
        with open(path, encoding="utf-8") as fh:
            rows = [json.loads(line) for line in fh if line.strip()]
        source = f"{repo}/{file} (hf_hub_download)"
    except Exception as first:  # noqa: BLE001 - try the datasets library next
        rows, problems = None, [f"{file}: {first!r}"]
        for revision in ("refs/convert/parquet", None):
            try:
                from datasets import load_dataset

                rows = [dict(r) for r in load_dataset(repo, "python", split="train", revision=revision)]
                source = f"datasets.load_dataset({repo!r}, 'python', split='train', revision={revision!r})"
                break
            except Exception as e:  # noqa: BLE001
                problems.append(f"datasets (revision {revision}): {e!r}")
        if rows is None:
            raise RuntimeError("Could not download the training commits: " + " | ".join(problems)) from None
    if not rows:
        raise RuntimeError(f"No commits in {source}")
    missing = [f for f in COMMIT_FIELDS if f not in rows[0]]
    if missing:
        raise RuntimeError(f"The commits have no {missing}; they have {sorted(rows[0])}")
    return rows, source


def subject_of(row) -> str:
    """The commit's first line, without a full stop at the end."""
    text = (row.get("subject") or row.get("message") or "").strip()
    return (text.splitlines()[0] if text else "").strip().rstrip(" .!")


def row_licence_ok(row) -> bool:
    """True when the row has no licence field, or its licence(s) are all permissive."""
    if "license" not in row:
        return True
    value = row["license"]
    names = [value] if isinstance(value, str) else list(value or [])
    return bool(names) and all(str(n).strip().lower() in ROW_LICENCES for n in names)


def is_bug_fix(row) -> bool:
    """The subject or message says it fixes something, and the subject is not a merge, revert, version bump or docs."""
    subject = subject_of(row)
    return bool(FIX.search(subject) or FIX.search(row.get("message") or "")) and not NOT_A_FIX.search(subject)


def is_test_file(row) -> bool:
    return bool(TEST_FILE.search((row.get("new_file") or row.get("old_file") or "").replace("\\", "/")))


def changed_lines(old: str, new: str) -> int:
    """Lines removed + lines added between the two files."""
    diff = difflib.unified_diff(old.splitlines(), new.splitlines(), lineterm="", n=0)
    return sum(1 for line in diff if line[:1] in "+-" and not line.startswith(("+++", "---")))


def question(row) -> str:
    """The user's message: the file before the fix, then the instruction (the same style as the HumanEvalFix prompt)."""
    old = row["old_contents"].strip("\n")
    return f"```python\n{old}\n```\n\n" + TRAIN_INSTRUCTION.format(subject=subject_of(row))


def answer(row) -> str:
    """The reply the model learns: the whole file after the fix, in one python block."""
    return f"```python\n{row['new_contents'].strip(chr(10))}\n```"


def messages_for(row) -> list[dict]:
    return [{"role": "user", "content": question(row)}, {"role": "assistant", "content": answer(row)}]


def words(text: str) -> list[str]:
    """Lowercase \\w+ words: what decontamination compares."""
    return re.findall(r"\w+", (text or "").lower())


def ngrams(tokens: list[str], n: int = NGRAM):
    return (" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def signature(line: str) -> str:
    """A def line without any spaces, so formatting does not matter."""
    return re.sub(r"\s+", "", line)


DEF_LINE = re.compile(r"^[ \t]*(?:async[ \t]+)?def[ \t]+\w+[ \t]*\(.*$", re.MULTILINE)


class Contamination:
    """What the HumanEvalFix tasks contain: every 10-word sequence and every entry point's def line."""

    def __init__(self, tasks, n: int = NGRAM) -> None:
        self.n, self.grams, self.signatures = n, {}, {}
        for task in tasks:
            for field in DECONTAM_FIELDS:
                for gram in ngrams(words(task.get(field) or ""), n):
                    self.grams.setdefault(gram, task["task_id"])
            source = (task.get("declaration") or "") + "\n" + (task.get("prompt") or "")
            name = re.escape(task["entry_point"])
            for line in re.findall(rf"^[ \t]*def[ \t]+{name}[ \t]*\(.*$", source, re.MULTILINE):
                self.signatures[signature(line)] = task["task_id"]

    def match(self, row) -> tuple[str, str] | None:
        """(task_id, why) when the row shares text with a task, else None."""
        text = "\n".join((row.get("old_contents") or "", row.get("new_contents") or "", row.get("subject") or "",
                          row.get("message") or ""))
        for line in DEF_LINE.findall(text):
            task_id = self.signatures.get(signature(line))
            if task_id:
                return task_id, f"defines {line.strip()[:80]}"
        for gram in ngrams(words(text), self.n):
            task_id = self.grams.get(gram)
            if task_id:
                return task_id, f"shares {self.n} words: {gram[:80]}"
        return None


def select_rows(rows, count_tokens, max_len, tasks, log=print):
    """The rows to train on, and how many each filter dropped (in this order):

    licence (the row's own), not_a_fix (message), test_file, empty_or_same, code_fence (``` inside the file, which
    would break the answer block), too_many_changes (> MAX_CHANGED_LINES), duplicate (same before and after),
    too_long (> max_len tokens, question + answer), contaminated (shares text with HumanEvalFix).
    count_tokens(row) gives the length of the whole example in tokens.
    """
    counts = {"raw": len(rows), "licence": 0, "not_a_fix": 0, "test_file": 0, "empty_or_same": 0, "code_fence": 0,
              "too_many_changes": 0, "duplicate": 0, "too_long": 0, "contaminated": 0,
              "contaminated_by": {"def_line": 0, "10_words": 0}, "contaminated_examples": [], "kept": 0}
    seen, survivors = set(), []
    for row in rows:
        old, new = row.get("old_contents") or "", row.get("new_contents") or ""
        if not row_licence_ok(row):
            counts["licence"] += 1
        elif not is_bug_fix(row):
            counts["not_a_fix"] += 1
        elif is_test_file(row):
            counts["test_file"] += 1
        elif not old.strip() or not new.strip() or old == new:
            counts["empty_or_same"] += 1
        elif "```" in old or "```" in new:
            counts["code_fence"] += 1
        elif changed_lines(old, new) > MAX_CHANGED_LINES:
            counts["too_many_changes"] += 1
        elif (old, new) in seen:
            counts["duplicate"] += 1
        else:
            seen.add((old, new))
            survivors.append(row)
    log(f"{len(survivors)} of {len(rows)} commits are small bug fixes; measuring their length in tokens")
    short = []
    for row in survivors:
        # A file this many characters long is far over max_len tokens: no need to count.
        too_long = len(row["old_contents"]) + len(row["new_contents"]) > 16 * max_len or count_tokens(row) > max_len
        counts["too_long"] += too_long
        if not too_long:
            short.append(row)
    log(f"{len(short)} fit in {max_len} tokens; checking them against HumanEvalFix")
    check, kept = Contamination(tasks), []
    for row in short:
        found = check.match(row)
        if found is None:
            kept.append(row)
            continue
        counts["contaminated"] += 1
        counts["contaminated_by"]["def_line" if found[1].startswith("defines") else "10_words"] += 1
        if len(counts["contaminated_examples"]) < 20:
            counts["contaminated_examples"].append({"commit": row.get("commit"), "subject": subject_of(row)[:100],
                                                    "task_id": found[0], "why": found[1]})
    counts["kept"] = len(kept)
    return kept, counts


def pick(rows, n, seed=3407):
    """n rows in a shuffled order (the same order every time for the same seed)."""
    rows = list(rows)
    random.Random(seed).shuffle(rows)
    return rows[:n] if n else rows


def counts_lines(counts) -> list[str]:
    """The data filters as text lines."""
    if not counts:
        return ["(no data counts)"]
    lines = [f"{counts['raw']:7,d} commits (Python) in the dataset"]
    names = {"licence": "repository licence not permissive", "not_a_fix": "message is not a bug fix",
             "test_file": "only a test file", "empty_or_same": "file empty, or no change",
             "code_fence": "``` inside the file", "too_many_changes": f"more than {MAX_CHANGED_LINES} lines changed",
             "duplicate": "duplicate (same before and after)", "too_long": "too long in tokens",
             "contaminated": "decontamination: shares text with HumanEvalFix"}
    for key, words_ in names.items():
        lines.append(f"{-counts.get(key, 0):7,d} {words_}")
    by = counts.get("contaminated_by") or {}
    if counts.get("contaminated"):
        lines[-1] += f" ({by.get('10_words', 0)} by 10 words, {by.get('def_line', 0)} by a def line)"
    lines.append(f"{counts['kept']:7,d} kept")
    return lines


# ---- scoring (the same prompt, code extraction and tests as phase 1: fixeval.py) ----

def score(generate, tasks, entry, minutes, batch=4, save=lambda: None, log=print):
    """Score a model on the tasks, filling entry (a dict in the results) as it goes.

    generate(prompts) -> one {"reply", "tokens", "cut_off"} per prompt. Tasks go `batch` at a time; when the GPU runs
    out of memory the batch is halved and tried again. Stops when `minutes` have passed.
    """
    deadline = time.time() + minutes * 60
    entry.update(total=len(tasks), done=0, passed=0, pass_at_1=None, rate_done=None, batch=batch, cut_off=0,
                 no_code_block=0, tokens=0, generate_seconds=0.0, tokens_per_s=None, avg_seconds=None,
                 stopped_by_time=False, minutes=None, errors=[], tasks=[])
    started, i, failures_in_a_row = time.time(), 0, 0
    while i < len(tasks):
        if time.time() > deadline:
            entry["stopped_by_time"] = True
            log(f"   stopped at {minutes} minutes, after {entry['done']} of {len(tasks)} tasks")
            break
        group = tasks[i:i + batch]
        began = time.time()
        try:
            answers = list(generate([prompt_for(t) for t in group]))
            if len(answers) != len(group):
                raise RuntimeError(f"{len(answers)} answers for {len(group)} prompts")
        except Exception as e:  # noqa: BLE001
            if "out of memory" in repr(e).lower() and batch > 1:
                batch = max(1, batch // 2)
                entry["batch"] = batch
                entry["errors"].append(f"out of memory: batch now {batch}")
                log(f"   out of GPU memory: trying {batch} at a time")
                continue
            answers, why = None, f"generation failed: {e!r}"[:300]
            entry["errors"].append(f"{group[0]['task_id']}: {why}")
            failures_in_a_row += 1
        else:
            failures_in_a_row = 0
        seconds = time.time() - began
        entry["generate_seconds"] += seconds
        for k, task in enumerate(group):
            record = {"task_id": task["task_id"], "passed": False}
            if answers is None:
                record["why"] = why
            else:
                a = answers[k]
                reply = a.get("reply") or ""
                has_block = "```" in without_thinking(reply)
                passed, why_ = run_program(program_for(extract_code(reply), task))
                record.update(passed=passed, why=why_ if reply.strip() else "no answer", reply=reply,
                              tokens=a.get("tokens") or 0, cut_off=bool(a.get("cut_off")), code_block=has_block,
                              seconds=round(seconds / len(group), 2))
                entry["tokens"] += record["tokens"]
                entry["cut_off"] += record["cut_off"]
                entry["no_code_block"] += not has_block
            entry["tasks"].append(record)
            entry["done"] += 1
            entry["passed"] += record["passed"]
            log(f"   {entry['done']:3d}/{len(tasks)} {task['task_id']:16s} {'pass' if record['passed'] else 'FAIL'}  "
                f"{'' if record['passed'] else record['why'][:80]}")
        i += len(group)
        summarise(entry, started)
        save()
        if failures_in_a_row >= 3:
            entry["errors"].append("3 batches in a row failed: stopped")
            log("   3 batches in a row failed: stopped")
            break
    summarise(entry, started)
    save()
    return entry


def summarise(entry, started) -> None:
    done, total = entry["done"], entry["total"]
    entry["pass_at_1"] = round(entry["passed"] / total, 4) if total and done == total else None
    entry["rate_done"] = round(entry["passed"] / done, 4) if done else None
    entry["generate_seconds"] = round(entry["generate_seconds"], 1)
    entry["tokens_per_s"] = round(entry["tokens"] / entry["generate_seconds"], 1) if entry["generate_seconds"] else None
    entry["avg_seconds"] = round(entry["generate_seconds"] / done, 2) if done else None
    entry["minutes"] = round((time.time() - started) / 60, 1)


def flips(before, after) -> dict:
    """Tasks scored in both: which the training fixed (fail → pass) and which it broke (pass → fail)."""
    was = {t["task_id"]: t["passed"] for t in (before or {}).get("tasks") or []}
    now = {t["task_id"]: t["passed"] for t in (after or {}).get("tasks") or []}
    both = [task_id for task_id in now if task_id in was]
    return {"both_done": len(both), "before_passed": sum(was[t] for t in both),
            "after_passed": sum(now[t] for t in both),
            "now_fixed": [t for t in both if now[t] and not was[t]],
            "now_broken": [t for t in both if was[t] and not now[t]]}


# ---- training progress ----

def note_progress(train, step, max_steps, logs, seconds) -> None:
    """Called at each training log: keep the loss and the time, and how long all the steps would take."""
    entry = {"step": step, **{k: round(v, 4) if isinstance(v, float) else v for k, v in (logs or {}).items()}}
    train.setdefault("log", []).append(entry)
    train.update(steps=step, planned_steps=max_steps, minutes=round(seconds / 60, 1))
    if "loss" in (logs or {}):
        train["last_loss"] = round(float(logs["loss"]), 4)
    if step and max_steps:
        train["projected_minutes"] = round(seconds / step * max_steps / 60, 1)


# ---- the report ----

def rate(entry):
    """(passed, done, total, share) of one score; share is of all tasks when all were done, else of those done."""
    entry = entry or {}
    share = entry.get("pass_at_1") if entry.get("pass_at_1") is not None else entry.get("rate_done")
    return entry.get("passed") or 0, entry.get("done") or 0, entry.get("total") or 0, share


def report_lines(results) -> list[str]:
    """Before → after on HumanEvalFix, the tasks that changed, and how it compares with phase 1."""
    lines = [f"{results.get('benchmark', 'HumanEvalFix')}: {results.get('base', '?')}, before and after training",
             f"{'':8s} {'passed':>13s} {'%':>5s} {'tok/s':>6s}  note"]
    for name in ("before", "after"):
        entry = results.get(name) or {}
        passed, done, total, share = rate(entry)
        note = []
        if not entry.get("done"):
            note.append("not scored")
        elif entry.get("stopped_by_time"):
            note.append(f"stopped at the time limit after {done} of {total} tasks")
        elif done < total:
            note.append(f"stopped after {done} of {total} tasks")
        if entry.get("cut_off"):
            note.append(f"{entry['cut_off']} cut off")
        if entry.get("errors"):
            note.append(f"{len(entry['errors'])} error" + ("s" if len(entry["errors"]) > 1 else ""))
        count = f"{passed}/{done}" + ("" if done == total else f" of {total}")
        lines.append(f"{name:8s} {count:>13s} {'-' if share is None else f'{share:.0%}':>5s} "
                     f"{entry.get('tokens_per_s') or '-':>6}  {'; '.join(note)}".rstrip())
    f = results.get("flips")
    if f and f.get("both_done"):
        change = f["after_passed"] - f["before_passed"]
        lines.append(f"On the {f['both_done']} tasks scored both times: {f['before_passed']} → {f['after_passed']} "
                     f"({change:+d}); now fixed {len(f['now_fixed'])}, now broken {len(f['now_broken'])}")
        if f["now_fixed"]:
            lines.append("   now fixed:  " + ", ".join(f["now_fixed"]))
        if f["now_broken"]:
            lines.append("   now broken: " + ", ".join(f["now_broken"]))
    name, target = TARGET
    _, done, total, share = rate(results.get("after"))
    full = not results.get("smoke") and (results.get("settings") or {}).get("EVAL_LIMIT") is None
    if share is not None and done == total and full:  # only a full run (all tasks) is compared
        verdict = "yes" if share > target else "no"
        lines.append(f"Beats phase 1's best ({name}, {target:.0%} with Ollama)? {verdict}: {share:.1%}")
    lines.append("(% = passed of all tasks when all were done, else of the tasks done; greedy pass@1)")
    return lines


RESULTS_FILE = 'coder_train.json'

SETTINGS = {k: globals()[k] for k in ("BASE", "MAX_TRAIN", "MAX_LEN", "EPOCHS", "EVAL_LIMIT", "SMOKE", "TRAIN_MINUTES",
                                      "EVAL_MINUTES", "EVAL_BATCH", "MAX_NEW_TOKENS", "MAX_STEPS", "SEED")}
RESULTS = {"phase": 2, "base": BASE, "smoke": SMOKE, "settings": SETTINGS, "benchmark": BENCHMARK,
           "eval_dataset": DATASET, "eval_licence": LICENCE, "eval_setting": EVAL_SETTING, "eval_source": None,
           "instruction": INSTRUCTION, "train_instruction": TRAIN_INSTRUCTION, "train_dataset": TRAIN_DATASET,
           "licence": None, "data": {"source": None, "fields": None, "example": None, "counts": None, "used": None},
           "train": {"examples": None, "steps": None, "planned_steps": None, "loss": None, "minutes": None,
                     "stopped_by_time": False, "log": []},
           "before": {}, "after": {}, "flips": None, "lora_dir": None, "export": NO_EXPORT, "phase1": PHASE1,
           "versions": None, "gpu": None, "errors": []}


def save_results():
    """Write RESULTS to coder_train.json. Done after every stage, so a stop part-way loses nothing."""
    if os.path.isdir(WORKING_DIR):
        with open(os.path.join(WORKING_DIR, "coder_train.json"), "w", encoding="utf-8") as fh:
            json.dump(RESULTS, fh, ensure_ascii=False, indent=1)


def failed(stage, error):
    """Record a failed stage and go on (Kaggle's Run all stops at the first error that is not caught)."""
    message = f"{stage}: {error!r}" if isinstance(error, Exception) else f"{stage}: {error}"
    print("FAILED", message)
    RESULTS["errors"].append(message)
    save_results()


def sh(command, timeout=120):
    """What a shell command printed, or "" when the command is missing or fails."""
    try:
        return subprocess.run(command, shell=True, capture_output=True, text=True, timeout=timeout).stdout.strip()
    except Exception:
        return ""

In [ ]:
#@title 5. Load the base model in 4 bits and attach LoRA (about 5–10 minutes)
from unsloth import FastLanguageModel, is_bfloat16_supported  # first: Unsloth speeds up what it imports after it
from unsloth.chat_templates import train_on_responses_only
import importlib.metadata as md
import torch


def version(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None


try:
    RESULTS["versions"] = {p: version(p) for p in ("unsloth", "torch", "transformers", "trl", "peft", "bitsandbytes")}
    RESULTS["gpu"] = torch.cuda.get_device_name(0)
    print(RESULTS["versions"], RESULTS["gpu"])
    model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE, max_seq_length=MAX_LEN, load_in_4bit=True)
    # LoRA: train a small add-on (about 0.5% of the weights). Its new part starts at zero, so until training the
    # model answers exactly as the base model does: the BEFORE score is the base model's.
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=SEED,
    )
except Exception as e:
    failed("load the model", e)
    raise
save_results()

eos = model.generation_config.eos_token_id
STOP_IDS = {i for i in [tokenizer.eos_token_id, tokenizer.pad_token_id, *(eos if isinstance(eos, list) else [eos])]
            if i is not None}


def generate_replies(prompts):
    """Greedy answers to a batch of prompts (each the only user message, in the base model's chat format)."""
    texts = [tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                           add_generation_prompt=True) for p in prompts]
    side, tokenizer.padding_side = tokenizer.padding_side, "left"  # batched answers need padding on the left
    try:
        inputs = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None,
                                 top_p=None, top_k=None, repetition_penalty=1.0, use_cache=True,
                                 pad_token_id=tokenizer.pad_token_id)
    except Exception:
        torch.cuda.empty_cache()
        raise
    finally:
        tokenizer.padding_side = side
    answers = []
    for row in out[:, inputs["input_ids"].shape[1]:].tolist():
        n = next((k + 1 for k, t in enumerate(row) if t in STOP_IDS), len(row))
        answers.append({"reply": tokenizer.decode(row[:n], skip_special_tokens=True), "tokens": n,
                        "cut_off": n >= MAX_NEW_TOKENS and row[n - 1] not in STOP_IDS})
    return answers

In [ ]:
#@title 6. Training data: open bug-fix commits, filtered and decontaminated
try:
    HUMANEVAL, RESULTS["eval_source"] = load_tasks()  # the test tasks: needed here to keep them out of training
    EVAL_TASKS = HUMANEVAL[:EVAL_LIMIT] if EVAL_LIMIT else HUMANEVAL
    RESULTS["licence"] = dataset_licence()
    print("Licence of", TRAIN_DATASET, RESULTS["licence"])
    commits, RESULTS["data"]["source"] = load_commits()
    RESULTS["data"]["fields"] = sorted(commits[0])
    RESULTS["data"]["example"] = {k: str(v)[:300] for k, v in commits[0].items()}
    print(f"{len(commits):,} commits from {RESULTS['data']['source']}")
    print("Fields:", ", ".join(RESULTS["data"]["fields"]))
    for field, value in RESULTS["data"]["example"].items():  # the first commit, field by field
        print(f"--- {field}:\n{value}")
    save_results()

    def count_tokens(row):
        chat = tokenizer.apply_chat_template(messages_for(row), tokenize=False)
        return len(tokenizer(chat, add_special_tokens=False)["input_ids"])

    kept, RESULTS["data"]["counts"] = select_rows(commits, count_tokens, MAX_LEN, HUMANEVAL)
    del commits
    TRAIN_ROWS = pick(kept, MAX_TRAIN, SEED)
    RESULTS["data"]["used"] = len(TRAIN_ROWS)
except Exception as e:
    failed("training data", e)
    raise
print("\n".join(counts_lines(RESULTS["data"]["counts"])))
print(f"Training on {len(TRAIN_ROWS)} of them; decontamination dropped {RESULTS['data']['counts']['contaminated']}")
for example in RESULTS["data"]["counts"]["contaminated_examples"][:5]:
    print("   dropped:", example)
print("--- One training example, as the model sees it:")
for message in messages_for(TRAIN_ROWS[0]):
    print(f"[{message['role']}]\n{message['content'][:1500]}")
save_results()

In [ ]:
#@title 7. Score the model BEFORE training (HumanEvalFix, up to EVAL_MINUTES)
print(f"BEFORE: {len(EVAL_TASKS)} tasks, {EVAL_BATCH} at a time")
FastLanguageModel.for_inference(model)
try:
    score(generate_replies, EVAL_TASKS, RESULTS["before"], EVAL_MINUTES, EVAL_BATCH, save_results)
except Exception as e:  # record it; the training still runs
    failed("BEFORE score", e)
finally:
    FastLanguageModel.for_training(model)
save_results()
print("\n".join(report_lines(RESULTS)))

In [ ]:
#@title 8. Train (QLoRA, up to TRAIN_MINUTES) and save the LoRA
from datasets import Dataset
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer


class Progress(TrainerCallback):
    """Writes the loss to coder_train.json at each log, and stops the training after TRAIN_MINUTES."""

    def __init__(self):
        self.started = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        note_progress(RESULTS["train"], state.global_step, state.max_steps, logs, time.time() - self.started)
        save_results()

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() - self.started > TRAIN_MINUTES * 60 and not control.should_training_stop:
            RESULTS["train"]["stopped_by_time"] = True
            print(f"Stopping: {TRAIN_MINUTES} minutes have passed (step {state.global_step} of {state.max_steps})")
            control.should_training_stop = True
        return control


dataset = Dataset.from_list([{"text": tokenizer.apply_chat_template(messages_for(r), tokenize=False)}
                             for r in TRAIN_ROWS])
RESULTS["train"]["examples"] = len(dataset)
progress = Progress()
try:
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=dataset,
        callbacks=[progress],
        args=SFTConfig(
            dataset_text_field="text", max_length=MAX_LEN,
            per_device_train_batch_size=1, gradient_accumulation_steps=8,
            num_train_epochs=EPOCHS, max_steps=MAX_STEPS or -1,
            learning_rate=2e-4, warmup_steps=10, lr_scheduler_type="linear",
            optim="adamw_8bit", weight_decay=0.01, logging_steps=LOGGING_STEPS, seed=SEED,
            fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
            output_dir="/tmp/coder-outputs", save_strategy="no", report_to="none",
        ),
    )
    # Learn only the answers (the fixed file), not the questions.
    trainer = train_on_responses_only(
        trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n",
    )
    stats = trainer.train()
    RESULTS["train"].update(steps=trainer.state.global_step, planned_steps=trainer.state.max_steps,
                            loss=round(stats.metrics["train_loss"], 4),
                            minutes=round((time.time() - progress.started) / 60, 1))
except Exception as e:
    RESULTS["train"]["minutes"] = round((time.time() - progress.started) / 60, 1)
    failed("training", e)
    raise
print(f"Trained {RESULTS['train']['steps']} of {RESULTS['train']['planned_steps']} steps "
      f"({len(dataset)} examples) in {RESULTS['train']['minutes']} min, loss {RESULTS['train']['loss']}")

try:  # the LoRA add-on (a few hundred MB), in the Output tab
    model.save_pretrained(LORA_DIR)
    tokenizer.save_pretrained(LORA_DIR)
    RESULTS["lora_dir"] = LORA_DIR
    print("Saved the LoRA add-on to", LORA_DIR)
except Exception as e:
    failed("save the LoRA", e)
save_results()

In [ ]:
#@title 9. Score the model AFTER training
torch.cuda.empty_cache()
print(f"AFTER: {len(EVAL_TASKS)} tasks, {EVAL_BATCH} at a time")
FastLanguageModel.for_inference(model)
try:
    score(generate_replies, EVAL_TASKS, RESULTS["after"], EVAL_MINUTES, EVAL_BATCH, save_results)
except Exception as e:
    failed("AFTER score", e)
RESULTS["flips"] = flips(RESULTS["before"], RESULTS["after"])
save_results()

In [ ]:
#@title 10. Results: before → after
print("\n".join(report_lines(RESULTS)))
print("Training:", {k: v for k, v in RESULTS["train"].items() if k != "log"})
print("Data:", RESULTS["data"]["used"], "examples;", RESULTS["licence"])
print(RESULTS["export"])
for error in RESULTS["errors"]:
    print("Error:", error)
save_results()
print("Saved:", os.path.join(WORKING_DIR, RESULTS_FILE))

## What next
If AFTER beats BEFORE clearly (more tasks now fixed than now broken), train on more commits (`MAX_TRAIN`) or for
longer. To use the model in Ollama, the LoRA (`coder-lora`, Output tab) must be merged into the base model and
exported as a GGUF file: that needs more memory than Kaggle gives safely for a 14B model, so it is not done here.
`coder_train.json` keeps every reply, so the tasks that changed can be read.